# Country Intelligence System using PCA and K-Means
End-to-end notebook for country segmentation using the Unsupervised Learning on Country Data dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [ ]:
# Load dataset
df = pd.read_csv('Country-data.csv')
df.head()

In [ ]:
# Basic preprocessing
country_names = df['country']
X = df.drop('country', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

In [ ]:
# PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)
print('Components:', pca.n_components_)
print('Explained Variance:', pca.explained_variance_ratio_.sum())

In [ ]:
# Elbow Method
inertia = []
for k in range(2,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca)
    inertia.append(km.inertia_)

plt.plot(range(2,11), inertia, marker='o')
plt.xlabel('Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
# Final KMeans Model
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_pca)

df['Cluster'] = clusters
print('Silhouette Score:', silhouette_score(X_pca, clusters))
df.head()

In [ ]:
# Visualization
pca2 = PCA(n_components=2)
X_vis = pca2.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_vis[:,0], X_vis[:,1], c=clusters)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Country Clusters')
plt.show()

In [ ]:
# Countries requiring highest attention
cluster_summary = df.groupby('Cluster').mean(numeric_only=True)
print(cluster_summary)

df.sort_values('income').head(20)[['country','income','Cluster']]